# LIBRARIES IMPORTATION

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.stats.mstats import winsorize

import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.set_option("display.max.column", 20)
# Display revenue values in full
pd.options.display.float_format = '{:,.0f}'.format

# DATA IMPORTATION AND PROFILING

In [ ]:
customer = pd.read_csv(r"C:\Users\DELL\OIBSIP\DataAnalytics-L1-EDARetailSales\customers.csv")

In [ ]:
customer.head()

In [ ]:
cus_df = customer.copy()

In [ ]:
cus_df.head()

In [ ]:
cus_df.shape

In [ ]:
cus_df.info()

After checking details about this dataset, here is what i identified:
1. There are 5,000 enteries and 12 columns.
2. There are missing values in about 10 columns except from customer_id and preferred_channel column.  
3. Mismatched datatype in age, zip code and registration_date columns. Age is an object instead of an integer, registration_date is an object but it's supposed to be a datetime datatype, zip code is suppose to be a string.

# Data Cleaning

In [ ]:
duplicate_values = cus_df.duplicated().sum()
duplicate_values

In [ ]:
missing_values = cus_df.isna().sum()
missing_values

In [ ]:
# Dropping the columns that won't be needed for analysis
cus_df.drop(columns = ["full_name", "email", "street_address", "zip_code", "registration_date"], inplace = True)

In [ ]:
cus_df["age"].describe()

In [ ]:
g = sns.boxplot(y = "age", data = cus_df)
g.set_title("Customer Age Distribution", fontsize = 20, y = 1.02)
g.set_ylabel("Age")
plt.show()

Filling missing age with median, since there are outliers and the data is skewed because median don't have an effect on extreme values.

In [ ]:
cus_df["age"].fillna(cus_df["age"].median(), inplace = True)

In [ ]:
cus_df["age"] = cus_df["age"].astype("int64")

In [ ]:
cus_df["gender"].value_counts()

In [ ]:
cus_df["gender"].fillna("Female", inplace = True)

In [ ]:
cus_df["city"].unique()

In [ ]:
cus_df.dropna(subset = "city", inplace = True)

In [ ]:
cus_df["state"].unique()

In [ ]:
cus_df["state"].value_counts()

In [ ]:
cus_df.dropna(subset = "state", inplace = True)

In [ ]:
cus_df["preferred_channel"].unique()

In [ ]:
cus_df["preferred_channel"].value_counts(normalize = True)

In [ ]:
cus_df["preferred_channel"].value_counts()

In [ ]:
cus_df.dropna(subset = "preferred_channel", inplace = True)

### Missing Value Treatment

Missing values were handled based on the type and importance of each variable. **Age (118 missing values)** was filled using the **median**, since Age is numerical and the median is less affected by extreme values. **Gender (112 missing values)** was filled using the **mode**, as it is a categorical variable.

For **City (97), State (93), and Preferred Channel (114)**, the affected rows were dropped because the number of missing records was relatively small and there was no reliable information to determine the correct values. This avoided introducing incorrect or assumed information into the dataset and helped maintain data accuracy.


In [ ]:
# First convert phone number to a string.
cus_df["phone"] = cus_df["phone"].astype("string")

In [ ]:
# Extract extension
cus_df['extension'] = cus_df['phone'].str.extract(r'[xX](\d+)', expand = False)

In [ ]:
# Remove extension
cus_df['phone_clean'] = cus_df['phone'].str.replace(r'[xX]\d+', '', regex = True)

In [ ]:
# Remove brackets, spaces, dots, hyphens, +
cus_df['phone_clean'] = cus_df['phone_clean'].str.replace(r'\D', '', regex = True)

In [ ]:
# Turn empty strings into missing values
cus_df['phone_clean'] = cus_df['phone_clean'].replace('', pd.NA)

In [ ]:
# Validate standard 10-digit numbers
cus_df['phone_valid'] = cus_df['phone_clean'].str.fullmatch(r'\d{10}')

In [ ]:
# Handle 11-digit numbers beginning with 1
cus_df.loc[
    cus_df['phone_clean'].str.fullmatch(r'1\d{10}', na = False), 'phone_clean'] = cus_df['phone_clean'].str[1:]

In [ ]:
# Handle 13-digit numbers beginning with 001
cus_df.loc[
    cus_df['phone_clean'].str.fullmatch(r'001\d{10}', na = False),'phone_clean'] = cus_df['phone_clean'].str[3:]

In [ ]:
# Anything that isn't exactly 10 digits becomes missing
cus_df.loc[
    ~cus_df['phone_clean'].str.fullmatch(r'\d{10}', na = False), 'phone_clean'] = pd.NA

In [ ]:
cus_df['phone_clean'].str.len().value_counts(dropna = False)

In [ ]:
cus_df[cus_df['phone_clean'].isna()][['phone', 'phone_clean']]

In [ ]:
cus_df['phone_clean'] = cus_df['phone_clean'].str.replace(
    r'(\d{3})(\d{3})(\d{4})', r'\1-\2-\3', regex = True)

This code takes the values in the phone_clean column and **reformats 10-digit phone numbers into the XXX-XXX-XXXX format**. The regular expression r'(\d{3})(\d{3})(\d{4})' splits the phone number into three groups: the first 3 digits, the next 3 digits, and the final 4 digits. Then r'\1-\2-\3' puts those groups back together with hyphens between them. For example, 0801234567 would become 080-123-4567. regex=True tells Pandas to treat the first argument as a regular expression.


In [ ]:
cus_df['preferred_channel'] = cus_df['preferred_channel'].str.title()
# To convert values to proper case.

In [ ]:
cus_df.head()

In [ ]:
cus_df.info()

In [ ]:
# Let drop phone number column, i only cleaned it because i want to learn how to clean phone number.
cus_df.drop(columns = ["phone", "phone_clean", "phone_valid", "extension"], inplace = True)

In [ ]:
cus_df.info()

In [ ]:
cus_df["gender"].unique()

In [ ]:
cus_df["city"].unique()

In [ ]:
cus_df["state"].unique()

In [ ]:
cus_df["preferred_channel"].unique()

In [ ]:
trans = pd.read_csv(r"C:\Users\DELL\OIBSIP\DataAnalytics-L1-EDARetailSales\transactions.csv")

In [ ]:
trans_df = trans.copy()

In [ ]:
trans_df.head()

In [ ]:
trans_df.shape

In [ ]:
trans_df.info()

The dataset comprises **32,295 transaction records and 10 variables**, including seven categorical and three numerical variables. The transaction_id, customer_id, and transaction_date fields are complete, while the remaining variables contain relatively small proportions of missing values. Missing observations range from **611 in discount_applied to 686 in product_category**, representing less than 3% of the dataset. Overall, the dataset is largely complete and suitable for further analysis.

In [ ]:
trans_df.isna().sum()

In [ ]:
# Inspect this column
trans_df["product_name"].head(20)

In [ ]:
# Inspect this column
trans_df["product_category"].head(10)

In [ ]:
trans_df[trans_df['product_category'].isna()][['product_name']]

In [ ]:
trans_df.groupby('product_name')['product_category'].agg(
    lambda x: x.dropna().unique()
).loc[['Audio-Technica Turntable', 'Blender', 'Steam Deck', 'Wall Art', 'Asus ROG', 
       'iPhone 13', 'Cookware Set', 'Dell XPS 15', 'Table Lamp', 'Dishwasher']]

In [ ]:
category_map = trans_df.groupby('product_name')['product_category'].first()

trans_df['product_category'] = trans_df['product_category'].fillna(
    trans_df['product_name'].map(category_map))

In [ ]:
trans_df['product_category'].isna().sum()

The product category that are still missing indicate that the product name is missing.

In [ ]:
trans_df['quantity'].describe()

In [ ]:
g = sns.boxplot(y = "quantity", data = trans_df)
g.set_title("Product Quantity Distribution", fontsize = 20, y = 1.02)
g.set_ylabel("Quantity")
plt.show()

In [ ]:
trans_df["quantity"].fillna(trans_df["quantity"].median(), inplace = True)

I used the **median** because the quantity data is **skewed and contains extreme values**, such as 50. The median is less affected by these outliers than the mean.


In [ ]:
trans_df["quantity"] = trans_df["quantity"].astype("int64")

In [ ]:
trans_df["price"].describe()

In [ ]:
g = sns.boxplot(y = "price", data = trans_df)
g.set_title("Price Distribution", fontsize = 20, y = 1.02)
g.set_ylabel("Price")
plt.show()

In [ ]:
trans_df["price"].fillna(trans_df["price"].median(), inplace = True)

In [ ]:
trans_df['discount_applied'].describe()

In [ ]:
g = sns.boxplot(y = "discount_applied", data = trans_df)
g.set_title("Discount Distribution", fontsize = 20, y = 1.02)
g.set_ylabel("Discount ")
plt.show()

In [ ]:
# I will be replacing NANs with 0 because it means there was no discount and 0 is actually the mean.
trans_df["discount_applied"].fillna(trans_df["discount_applied"].median(), inplace = True)

In [ ]:
trans_df["discount_applied"] = trans_df["discount_applied"].astype("int64")

In [ ]:
trans_df.isna().sum()

In [ ]:
trans_df["store_location"].unique()

In [ ]:
trans_df["store_location"].value_counts()

In [ ]:
trans_df.dropna(subset = ["store_location"], inplace = True)

In [ ]:
trans_df.dropna(subset = ["payment_method"], inplace = True)

In [ ]:
trans_df.dropna(subset = ['product_category'], inplace = True)

In [ ]:
trans_df.dropna(subset = ['product_name'], inplace = True)

### Missing Value Treatment

Missing values were handled based on the variable type and reliability of the available data. **Price, Quantity, and Discount Applied** were filled using the **median** because these numerical variables contained outliers.

For **Product Name**, missing records were dropped. For **Product Category**, existing product names were mapped to their corresponding categories. Only 19 records remained unmatched and were dropped.

Missing **Store Location** and **Payment Method** records were also dropped because they were relatively few and could not be reliably determined.


In [ ]:
trans_df["transaction_date"] = pd.to_datetime(trans_df["transaction_date"], errors = "coerce")

In [ ]:
cus_df.info()

In [ ]:
cus_df.duplicated().sum()

In [ ]:
trans_df.info()

In [ ]:
trans_df.duplicated().sum()

In [ ]:
df = trans_df.merge(cus_df, on = "customer_id", how = "left")

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Investigating why we are missing values since the both dataset were both cleaned before merging.
df[df['gender'].apply(lambda x: isinstance(x, (int, float)))][
    ['age', 'gender', 'city', 'state', 'preferred_channel']
].head()

In [ ]:
df = df.dropna(subset =['age', 'gender', 'city', 'state', 'preferred_channel'])

In [ ]:
df[['age', 'gender', 'city', 'state', 'preferred_channel']].isna().sum()

Rows with missing customer information were removed because they represented only 5.5% of the dataset. Rather than assigning the same values to a large number of records, which could introduce bias and affect the reliability of the analysis, the incomplete records were excluded. This ensured that the remaining data was based on actual available information and supported more trustworthy business insights.

In [ ]:
df.duplicated().sum()

In [ ]:
df["age"].unique()

In [ ]:
df["age"] = df["age"].astype("int64")

In [ ]:
df["state"].unique()

In [ ]:
df["payment_method"].unique()

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include = 'number').columns
num_cols

In [ ]:
# Checking for outliers
for col in num_cols:
    plt.figure(figsize = (10, 5))
    plt.boxplot(x = df[col])
    plt.title(f'{col.capitalize()}, (Detecting outliers)', fontsize = 16, y = 1.04)
    plt.xlabel(col.capitalize())
    plt.tight_layout()
    plt.show()

In [ ]:
df[['quantity', 'price', 'age', 'discount_applied']].describe()

In [ ]:
for col in num_cols:
    df[col] = winsorize(df[col], limits = [0.05, 0.05])

Outlier Analysis: The descriptive statistics and boxplots revealed extreme values in Quantity, Price, and Age. For example, Quantity reaches a maximum of 50, while Price reaches 21,371.65. However, these observations were investigated and confirmed to represent legitimate values in the dataset rather than data-entry errors. Consequently, the outliers were retained to preserve the original characteristics of the data. The Discount Applied variable showed comparatively fewer extreme observations, with values ranging from 0 to 30.

In [ ]:
df["revenue"] = df["quantity"] * df["price"]

In [ ]:
df["year"] = df["transaction_date"].dt.year

In [ ]:
df["month"] = df["transaction_date"].dt.month_name()

In [ ]:
df["month"].unique()

In [ ]:
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]
df["month"] = pd.Categorical(df["month"], categories = month_order, ordered = True)

In [ ]:
df["quarter"] = df["transaction_date"].dt.quarter
df["quarter"] = "Q" + df["quarter"].astype(str)

In [ ]:
df.info()

# EDA

DESCRIPTIVE  STATISTICS

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include = 'number').columns
num_cols = [col for col in num_cols if col != 'year']
num_cols

In [ ]:
summary = pd.DataFrame({
    'Mean': df[num_cols].mean(),
    'Median': df[num_cols].median(),
    'Mode': [df[col].mode().iloc[0] for col in num_cols],
    'Standard Deviation': df[num_cols].std()
}, index = num_cols)

summary

Insight: The average quantity purchased is about 1, and most customers purchase 1 item, showing that customers generally make small purchases. The average price is 590, but the median price is lower at 401, suggesting that some expensive products are increasing the average. Most transactions have no discount, as both the median and mode are 0, although some customers receive higher discounts. The average customer age is about 35 years, with most customers being around this age. Coming from the price column, where some product are increasing the average, it does the same to the revenue.

TIME SERIES ANALYSIS

In [ ]:
df["revenue"]

In [ ]:
yearly_sales = df.groupby("year")["revenue"].sum()
yearly_sales

In [ ]:
plt.figure(figsize = (6, 3))
plt.plot(yearly_sales.index, yearly_sales.values, marker='o')
plt.xlabel('Year')
plt.ylabel('Total Sales')
plt.title('Yearly Sales Trend')
# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}")
)
plt.show()

In [ ]:
monthly_sales = df.groupby("month", observed = True)["revenue"].sum()
monthly_sales

In [ ]:
plt.figure(figsize = (6, 3))
plt.plot(monthly_sales.index, monthly_sales.values, marker="o")

plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.title("Monthly Sales Trend")
plt.xticks(rotation=45)
# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}")
)
plt.show()

In [ ]:
quarterly_sales = df.groupby("quarter", observed = True)["revenue"].sum()
quarterly_sales

In [ ]:
plt.plot(quarterly_sales.index, quarterly_sales.values, marker="o")

plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.title("Monthly Sales Trend")
plt.xticks(rotation=45)
# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}")
)
plt.show()

### Insight from Sales Trend

**Yearly:** Sales increased steadily from **692,785 in 2020** to **7,466,081 in 2024**, showing strong year-on-year growth. However, sales decreased to **1,031,550 in 2025**. This sharp decrease should be investigated further to determine whether it is due to lower sales or because the 2025 dataset does not cover the full year.

**Monthly:** Sales increased from **January to August**, followed by a decline in **September and October**. Sales then increased again in **November**. Sales were highest in **December (3,066,801)** and **November (2,900,460)**, while the lowest sales occurred in **March (1,155,325)**. Sales generally increased toward the end of the year, with a noticeable peak in **November and December**.

**Quarterly:** Revenue was highest in **Q4 (7,454,433)** and lowest in **Q2 (3,713,007)**. Revenue increased strongly in the second half of the year, with **Q3 and Q4** showing better performance than Q1 and Q2.

**Overall insight:** The fourth quarter generated the **highest sales**, suggesting that sales tend to be stronger toward the end of the year. This may indicate a possible seasonal pattern and could help the business plan inventory, marketing, and promotions for the final quarter.


CUSTOMER DEMOGRAPHICS

In [ ]:
plt.figure(figsize = (6, 3))
plt.hist(df["age"], bins = 10, alpha = 0.8, edgecolor = "black", linewidth = 1.2)
plt.tight_layout()
plt.title("Customer Age Distribution", fontsize = 20, pad = 10)
plt.xlabel("Age", fontsize = 14)
plt.ylabel("Number of customers", fontsize = 14)
plt.grid(axis = "y", linestyle = "--", alpha = 0.3)
plt.show()

### Observation

The age distribution shows that most customers are around **35 years old**, indicating that the business has a strong customer base within the adult age range.


In [ ]:
gender_counts = df["gender"].value_counts()

In [ ]:
bars = plt.bar(gender_counts.index, gender_counts.values)
for bar in bars:
    value = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2,
        value, f"{int(value):,}", ha = "center", va = "bottom")
plt.title("Number of Customers by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Customers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Observation

Female customers make up the largest gender group, with **14,463 customers**, followed by males with **13,105 customers**. Non-binary, Unknown, and Prefer not to say customers represent relatively smaller groups.


In [ ]:
bins = [0, 29, 59, 100]

labels = [
    "Young Adult",
    "Adult",
    "Older Adult"
]
df["age_group"] = pd.cut(df["age"], bins = bins, labels = labels, include_lowest = True)

In [ ]:
age_group_counts = df["age_group"].value_counts()

In [ ]:
bars = plt.bar(age_group_counts.index, age_group_counts.values)

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{int(value):,}",
        ha="center",
        va="bottom"
    )

plt.title("Number of Customers by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Customers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Observation

The **Adult group has the highest number of customers, with 20,575 records**, followed by Young Adults with 8,048 records. No customers were recorded in the Older Adult group, showing that the customer base is mainly concentrated among younger and middle-aged customers.


In [ ]:
# Revenue by Gender
gender_revenue = df.groupby("gender")["revenue"].sum().sort_values(ascending=False)
gender_revenue

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(gender_revenue.index, gender_revenue.values)

plt.title("Revenue by Gender")
plt.xlabel("Gender")
plt.ylabel("Revenue")
plt.xticks(rotation=45)

# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}")
)

plt.tight_layout()
plt.show()

### Insight from Revenue by Gender

**Gender:** Revenue was highest among **Female customers (10,793,208)**, followed by **Male customers (9,662,962)**. **Non-binary customers** and those who **preferred not to say** contributed a smaller share of total revenue.


In [ ]:
# Revenue by Age group
age_group_revenue = df.groupby("age_group")["revenue"].sum().sort_values(ascending=False)
age_group_revenue

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(age_group_revenue.index, age_group_revenue.values)

plt.title("Revenue by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Revenue")
plt.xticks(rotation=45)

# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}")
)

plt.tight_layout()
plt.show()

### Insight from Revenue by Age Group

**Age Group:** Revenue was highest among **Adults (15,123,336)**, while **Young Adults generated 6,090,201**. **Older Adults recorded no revenue** in the dataset.

In [ ]:
# Revenue by Age Group and Gender
age_gender_revenue = (
    df.groupby(["age_group", "gender"], observed=True)["revenue"].sum().unstack())
age_gender_revenue

In [ ]:
# Plot
ax = age_gender_revenue.plot(kind = "bar", figsize = (10, 6))

plt.title("Revenue by Age Group and Gender")
plt.xlabel("Age Group")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.legend(title="Gender")

# Display full revenue values on Y-axis
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

### Insight from Revenue by Age Group and Gender

**Age & Gender:** **Adult females generated the highest revenue (7,680,044)**, followed by **adult males (6,914,677)**. Among Young Adults, females also generated more revenue than males. Overall, **Adults contributed the largest share of revenue across all gender groups**.

# PRODUCT ANALYSIS

In [ ]:
top_10_product = df.groupby("product_name")["quantity"].sum().sort_values(ascending = False).head(10)
top_10_product

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(top_10_product.index, top_10_product.values)

plt.title("Top 10 Selling Product")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.xticks(rotation=45)

# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

### Insight from Product Quantity

**Product:** **iPhone 13 had the highest quantity sold (1,061 units)**, followed by **OnePlus 10 (1,036)** and **Xiaomi Mi 12 (1,027)**. **Bed Frame had the lowest quantity among the top 10 products (938 units)**.


In [ ]:
# Revenue by product category
category_revenue = df.groupby("product_category")["revenue"].sum().sort_values(ascending = False)
category_revenue

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(category_revenue.index, category_revenue.values)

plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation = 90)

# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

### Insight from Revenue by Product Category

**Product Category:** **Furniture generated the highest revenue (4,735,126)**, followed by **Smartphones (3,993,861)** and **Laptops (3,039,839)**. **Computer Accessories generated the lowest revenue (66,036)**. Overall, revenue was concentrated in **Furniture, Smartphones, and Laptops**.


In [ ]:
# Do the product that sells the most generates the most revenue
# Total quantity sold and total revenue for each product
product_analysis = (
    df.groupby('product_name')
      .agg(
          Total_Quantity_Sold = ('quantity', 'sum'),
          Total_Revenue = ('revenue', 'sum')
      )
      .sort_values('Total_Quantity_Sold', ascending = False)
)

product_analysis.head(10)

### Insight from Product Performance

**Product:** **iPhone 13 had the highest quantity sold (1,061 units)** and generated **847,855 in revenue**. However, **Bed Frame generated the highest revenue (1,023,041)** despite having only **938 units sold**, indicating a higher revenue per unit.


In [ ]:
df[num_cols].corr()

In [ ]:
corr = df[num_cols].corr()
sns.heatmap(corr, annot = True, cmap = "coolwarm", fmt = ".2f")

plt.title("Correlation Heatmap")
plt.xticks(rotation = 90)
plt.show()

### Insight from Correlation Analysis

**Correlation:** The correlation matrix shows a **very strong positive relationship between price and revenue (1.00)**. The other variables **quantity, discount applied, and age** show little to no correlation with revenue.

**Key finding:** Revenue in this dataset appears to be driven mainly by **price**, rather than quantity, discount, or customer age.


# REGION ANALYSIS

In [ ]:
# Revenue by state
state_revenue = df.groupby("state")["revenue"].sum().sort_values(ascending = False)
state_revenue

In [ ]:
plt.bar(state_revenue.index, state_revenue.values)
plt.xlabel("State")
plt.ylabel("Revenue")
plt.title('Revenue by State')
plt.xticks(rotation = 90)
# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

### Insight from Revenue by State

**Insight:** California is the **top-performing state**, recording about **3,896,917 Million**, followed by Texas with **2,844,377 Milllion**. In contrast, Massachusetts is the **least-performing state** with approximately **600.608 thousand units**. This shows a significant difference in sales volume, with California leading the overall performance among the states analyzed.


In [ ]:
# Total Orders by state
state_order = df.groupby("state")["quantity"].sum().sort_values(ascending = False)
state_order

In [ ]:
plt.bar(state_order.index, state_order.values)
plt.xlabel("State")
plt.ylabel("Quantity")
plt.title('Quantity by State')
plt.xticks(rotation = 90)
# Display full values on Y-axis
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

### Insight from Total Quantity by State

**Insight:** California recorded the **highest quantity at 6,563**, making it the top-performing state. Massachusetts had the **lowest quantity at 1,055**. Overall, California significantly outperformed the other states, while Massachusetts recorded the weakest sales volume among the states analyzed.


In [ ]:
state_summary = pd.DataFrame({
    "Quantity" : state_order,
    "Revenue" : state_revenue
})
state_summary

Overall, California shows the strongest performance across both quantity and revenue.

In [ ]:
california_city_summary = df[df["state"] == "California"].groupby("city").agg(
    total_revenue = ("revenue", sum),
    total_quantity = ("quantity", sum)
).sort_values('total_revenue', ascending = False)
california_city_summary

**Insight:** San Diego is the **top-performing city by revenue**, generating **855,505**, while Los Angeles has the **highest quantity** with **1,440 units**. San Francisco is the **least-performing city**, with **583,423 revenue** and **1,008 units**. Overall, San Diego leads in revenue, while Los Angeles leads in sales volume.


### Conclusion

Overall, the business had **strong sales growth from 2020 to 2024**, with the **fourth quarter having the highest sales**. Adults were the main customers, with **adult females generating the highest revenue**. **Furniture** was the highest-revenue category. The analysis also showed that a product with high sales quantity does not always generate the highest revenue.

**California was the best-performing state**, with **6,563 units and 3,896,917 in revenue**, while **Massachusetts had the lowest performance**, with **1,055 units and 600,608 in revenue**. In California, **San Diego had the highest revenue at 855,505**, while **Los Angeles had the highest quantity at 1,440 units**.

### Recommendations

1. **Prepare for Q4:** Increase stock and marketing before November and December to meet higher demand.

2. **Focus on high-revenue products:** Give more attention to **Furniture** and other products that bring in more revenue.

3. **Target adult customers:** Create more promotions for **adult customers**, especially adult females.

4. **Focus on strong markets:** Give more attention to top-performing states such as **California and Texas**, while finding ways to improve weaker states such as Massachusetts.

5. **Focus on top cities:** In California, consider investing more in **San Diego** because it has the highest revenue and **Los Angeles** because it has the highest quantity.


In [ ]:
df.to_csv("Cleaned_retail_data.csv", index = False)